# Qwen 3.5: Gated Delta Networks + MoE

This notebook replicates the architecture of **Qwen 3.5** (Alibaba, 2026) — a model that
introduces a **third paradigm** for sequence processing: **Gated Delta Networks**. While
DeepSeek-V3 uses attention (MLA) and Nemotron-3 uses Mamba (SSMs), Qwen 3.5 uses the
**delta rule** — a fixed-size state matrix that can be correctively updated per token.

Combined with 256-expert MoE and native multimodality, this creates a fundamentally
different architecture from anything else in this repo.

Key innovations:
- **Gated DeltaNet** — O(N) sequence processing via corrective state matrix updates
- **3:1 Hybrid** — 3 DeltaNet + 1 GQA attention, repeated
- **256-expert MoE** — 8 routed + 1 shared per token (3.5% activation)
- **Natively multimodal** — DeepStack ViT with Conv3d for video
- **1M context** — enabled by DeltaNet's linear complexity

**Reference:** Qwen Team, Alibaba Cloud (2026). *Qwen 3.5 Technical Report.*
https://github.com/QwenLM/Qwen3.5

In [ ]:
import sys, os

# In Colab, clone the repo so local imports (src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/07_Model_Replications")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time
import matplotlib.pyplot as plt
from src.utils.device import set_seed

set_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 1. Three Paradigms for Sequence Processing

| | Attention (DeepSeek-V3) | Mamba / SSM (Nemotron-3) | **DeltaNet (Qwen 3.5)** |
|---|---|---|---|
| State | Full KV cache (grows with N) | Fixed-size vector | **Fixed-size matrix** |
| Update rule | Store all K, V | $h = \bar{A}h + \bar{B}x$ | $W = \alpha W + \beta(v - Wk)k^T$ |
| Complexity | O(N²) | O(N) | **O(N)** |
| Precise recall | Excellent | Weak | **Good (corrective)** |
| Parallelizable | Yes | Yes (scan) | **Yes (chunk-wise)** |
| Can overwrite old info | Yes (re-attend) | Difficult | **Yes (delta correction)** |

DeltaNet occupies a unique middle ground: it has O(N) complexity like Mamba, but its
state matrix can store and **correct** key-value associations like attention.

## 2. Architecture at a Glance

| | Qwen 3.5 397B-A17B | Qwen 3.5 35B-A3B | DeepSeek-V3 | Nemotron-3 Super |
|---|---|---|---|---|
| Total params | 397B | 35B | 671B | 120B |
| Active/token | 17B | 3B | 37B | 12B |
| Sequence layers | GatedDeltaNet | GatedDeltaNet | MLA (attention) | Mamba-2 (SSM) |
| Attention layers | Every 4th (GQA) | Every 4th (GQA) | All (MLA) | Every 4th |
| Experts | 512 (10+1 active) | 256 (8+1 active) | 256 (8+1 active) | LatentMoE |
| Context | 1M | 1M | 128K | 1M |
| Multimodal | Native (DeepStack ViT) | Text only | Text only | Text only |

## 3. The Delta Rule

The delta rule comes from classical neural network learning theory. Instead of storing
all tokens (attention) or using a recurrent hidden state (Mamba), DeltaNet maintains
a **state matrix** $W$ that maps keys to values:

$W \leftarrow W + \alpha(v_t - Wk_t)k_t^T$

The update term $(v_t - Wk_t)$ is the **delta** — the difference between:
- $v_t$: what the value *should* be for key $k_t$
- $Wk_t$: what the state matrix currently *predicts* for key $k_t$

**This is corrective:** If the state already has the right association for this key,
the delta is zero and nothing changes. If it's wrong, the delta corrects it.

## 4. Why Delta > Naive Linear Attention?

**Naive linear attention** just accumulates:

$W \leftarrow W + v_t k_t^T$

Problem: old info never gets overwritten. The state matrix becomes a "smear of all past
information" with no way to correct mistakes or forget irrelevant tokens.

**The delta rule can correct old entries.** If key $k$ was originally stored with value
$v_{old}$, and we later see the same key with value $v_{new}$:
- Naive: $W$ stores $v_{old} + v_{new}$ (corrupted)
- Delta: $W$ corrects toward $v_{new}$ (the delta overwrites the old association)

This is why DeltaNet is conceptually closer to attention than to Mamba — it maintains
a "learned KV cache" that can be updated.

In [ ]:
class DeltaNetLayer(nn.Module):
    """
    Core DeltaNet mechanism: state matrix updated via the delta rule.
    No gating yet — this is the basic version.
    """

    def __init__(self, d_model, d_state=32):
        super().__init__()
        self.d_state = d_state
        self.proj_q = nn.Linear(d_model, d_state, bias=False)
        self.proj_k = nn.Linear(d_model, d_state, bias=False)
        self.proj_v = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_state, d_model, bias=False)

    def forward(self, x):
        B, N, D = x.shape
        Q = self.proj_q(x)  # [B, N, d_state]
        K = self.proj_k(x)
        V = self.proj_v(x)

        # State matrix W: maps keys → values
        W = torch.zeros(B, self.d_state, self.d_state, device=x.device)
        outputs = []

        for t in range(N):
            k_t = K[:, t, :]  # [B, d_state]
            v_t = V[:, t, :]
            q_t = Q[:, t, :]

            # Delta rule: correct the state matrix
            predicted = torch.bmm(W, k_t.unsqueeze(-1)).squeeze(-1)  # Wk: what we predict
            delta = v_t - predicted  # difference from target
            W = W + torch.bmm(delta.unsqueeze(-1), k_t.unsqueeze(-2))  # outer product update

            # Output: query the state matrix
            y_t = torch.bmm(W, q_t.unsqueeze(-1)).squeeze(-1)  # Wq
            outputs.append(y_t)

        output = torch.stack(outputs, dim=1)  # [B, N, d_state]
        return self.out_proj(output)


# Demo: show the delta rule can correct associations
delta_net = DeltaNetLayer(d_model=16, d_state=8)
x = torch.randn(1, 10, 16)
out = delta_net(x)
print(f"DeltaNet: {x.shape} → {out.shape}")

# Compare with naive linear attention (just accumulates, no correction)
print("\nDelta rule vs naive accumulation:")
W_delta = torch.zeros(8, 8)
W_naive = torch.zeros(8, 8)

k = torch.randn(8)  # same key
v1 = torch.randn(8)  # original value
v2 = torch.randn(8)  # updated value

# Store k→v1
W_naive += torch.outer(v1, k)
W_delta += torch.outer(v1 - W_delta @ k, k)

# Update k→v2 (should overwrite v1)
W_naive += torch.outer(v2, k)
W_delta += torch.outer(v2 - W_delta @ k, k)

# Query: what does W think k maps to?
naive_recall = W_naive @ k
delta_recall = W_delta @ k

print(f"  Target (v2):      {v2[:4].numpy().round(3)}")
print(f"  Delta recall:     {delta_recall[:4].detach().numpy().round(3)} ← closer to v2")
print(f"  Naive recall:     {naive_recall[:4].detach().numpy().round(3)} ← corrupted (v1+v2)")
print(f"  Delta error:      {(delta_recall - v2).norm().item():.4f}")
print(f"  Naive error:      {(naive_recall - v2).norm().item():.4f}")

## 5. Gating for Selective Forgetting

The basic delta rule has no forgetting — the state matrix grows unbounded.
**Gated DeltaNet** adds two sigmoid-activated gates:

$W \leftarrow \text{diag}(\alpha_t) \cdot W + \beta_t \cdot (v_t - Wk_t) k_t^T$

- $\alpha_t = \sigma(W_\alpha x_t)$ — **forget gate**: decays old state (values < 1 cause forgetting)
- $\beta_t = \sigma(W_\beta x_t)$ — **input gate**: controls how much new info enters

**Why this matters:** Without gating, the state matrix eventually saturates —
it becomes "a smear of all past information." The gates let the model selectively
forget irrelevant history and focus on what's important.

In [ ]:
class GatedDeltaNet(nn.Module):
    """
    Gated DeltaNet: delta rule with input/forget gates.
    Used in Qwen 3.5 as the primary sequence processing layer.
    """

    def __init__(self, d_model, d_state=32):
        super().__init__()
        self.d_state = d_state

        self.proj_q = nn.Linear(d_model, d_state, bias=False)
        self.proj_k = nn.Linear(d_model, d_state, bias=False)
        self.proj_v = nn.Linear(d_model, d_state, bias=False)
        self.proj_alpha = nn.Linear(d_model, d_state, bias=True)   # forget gate
        self.proj_beta = nn.Linear(d_model, 1, bias=True)          # input gate
        self.out_proj = nn.Linear(d_state, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        B, N, D = x.shape

        Q = self.proj_q(x)
        K = self.proj_k(x)
        V = self.proj_v(x)
        alpha = torch.sigmoid(self.proj_alpha(x))  # forget gate [B, N, d_state]
        beta = torch.sigmoid(self.proj_beta(x))    # input gate [B, N, 1]

        W = torch.zeros(B, self.d_state, self.d_state, device=x.device)
        outputs = []

        for t in range(N):
            k_t = K[:, t, :]
            v_t = V[:, t, :]
            q_t = Q[:, t, :]
            alpha_t = alpha[:, t, :]  # [B, d_state]
            beta_t = beta[:, t, :]    # [B, 1]

            # Gated state update
            predicted = torch.bmm(W, k_t.unsqueeze(-1)).squeeze(-1)
            delta = v_t - predicted

            # Forget old state + add gated new info
            W = alpha_t.unsqueeze(-1) * W + beta_t.unsqueeze(-1) * torch.bmm(
                delta.unsqueeze(-1), k_t.unsqueeze(-2)
            )

            y_t = torch.bmm(W, q_t.unsqueeze(-1)).squeeze(-1)
            outputs.append(y_t)

        output = torch.stack(outputs, dim=1)
        return self.out_proj(output) + residual


# Demo
gdn = GatedDeltaNet(d_model=32, d_state=16)
x = torch.randn(1, 12, 32)
out = gdn(x)
print(f"GatedDeltaNet: {x.shape} → {out.shape}")
print(f"Parameters: {sum(p.numel() for p in gdn.parameters()):,}")

In [ ]:
# Compare scaling: DeltaNet (O(N)) vs Attention (O(N²))
d = 32
gdn_block = GatedDeltaNet(d_model=d, d_state=16)
attn_layer = nn.MultiheadAttention(d, num_heads=4, batch_first=True)

seq_lengths = [32, 64, 128, 256, 512]
gdn_times, attn_times = [], []

for N in seq_lengths:
    x = torch.randn(1, N, d)

    start = time.perf_counter()
    for _ in range(3):
        _ = gdn_block(x)
    gdn_times.append((time.perf_counter() - start) / 3 * 1000)

    start = time.perf_counter()
    for _ in range(3):
        _ = attn_layer(x, x, x)
    attn_times.append((time.perf_counter() - start) / 3 * 1000)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(seq_lengths, attn_times, 'o-', label='Attention (O(N²))', linewidth=2, color='tab:red')
ax.plot(seq_lengths, gdn_times, 's-', label='GatedDeltaNet (O(N))', linewidth=2, color='tab:purple')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('GatedDeltaNet vs Attention: Scaling', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Why Hybrid?

Same rationale as Nemotron-3, but with DeltaNet instead of Mamba:

DeltaNet's state matrix is lossy for **precise positional recall** — "the exact wording
of a function signature 400 tokens ago" can degrade through repeated state compression.

**Solution:** Use DeltaNet for 75% of layers (cheap, handles general processing) and
GQA attention every 4th layer for precise lookup.

The attention layers use **Grouped Query Attention** (2 KV heads per 16 Q heads) to
minimize the cost of the attention layers while maintaining recall quality.

In [ ]:
class GQABlock(nn.Module):
    """Grouped Query Attention block (used sparsely every 4th layer)."""

    def __init__(self, d_model, n_q_heads=8, n_kv_heads=2):
        super().__init__()
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_q_heads
        self.n_rep = n_q_heads // n_kv_heads  # how many Q heads share each KV head

        self.norm = nn.LayerNorm(d_model)
        self.W_q = nn.Linear(d_model, n_q_heads * self.head_dim, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.W_o = nn.Linear(n_q_heads * self.head_dim, d_model, bias=False)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        B, N, _ = x.shape

        Q = self.W_q(x).view(B, N, self.n_q_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # Repeat KV heads to match Q heads
        K = K.repeat_interleave(self.n_rep, dim=1)  # [B, n_q_heads, N, hd]
        V = V.repeat_interleave(self.n_rep, dim=1)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        out = torch.softmax(scores, dim=-1) @ V
        out = out.transpose(1, 2).contiguous().view(B, N, -1)

        return self.W_o(out) + residual


# Build hybrid backbone
class QwenMoELayer(nn.Module):
    """Fine-grained MoE with shared expert (Qwen-style)."""

    def __init__(self, d_model, d_ff, n_experts=32, top_k=4):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.norm = nn.LayerNorm(d_model)

        self.shared = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False), nn.SiLU(),
            nn.Linear(d_ff, d_model, bias=False)
        )
        self.gate = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff, bias=False), nn.SiLU(),
                nn.Linear(d_ff, d_model, bias=False)
            ) for _ in range(n_experts)
        ])

    def forward(self, x):
        residual = x
        x = self.norm(x)
        B, N, D = x.shape

        shared_out = self.shared(x)

        logits = self.gate(x)
        top_k_logits, indices = torch.topk(logits, self.top_k, dim=-1)
        weights = torch.softmax(top_k_logits, dim=-1)

        routed_out = torch.zeros_like(x)
        for k in range(self.top_k):
            idx = indices[:, :, k]
            w = weights[:, :, k].unsqueeze(-1)
            for i in range(self.n_experts):
                mask = (idx == i)
                if mask.any():
                    routed_out[mask] += (w[mask] * self.experts[i](x[mask])).squeeze(-2)

        return shared_out + routed_out + residual


# Visualize the block pattern
pattern = []
for cycle in range(2):  # 2 cycles of the 3:1 pattern
    pattern.extend(['DeltaNet', 'MoE', 'DeltaNet', 'MoE', 'DeltaNet', 'MoE', 'Attention', 'MoE'])

colors_map = {'DeltaNet': 'tab:purple', 'Attention': 'tab:red', 'MoE': 'tab:blue'}

fig, ax = plt.subplots(figsize=(14, 3))
for i, lt in enumerate(pattern):
    ax.barh(0, 1, left=i, color=colors_map[lt], edgecolor='white', linewidth=2)
    ax.text(i + 0.5, 0, f'{i}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=c, label=l) for l, c in colors_map.items()]
ax.legend(handles=handles, loc='upper right', fontsize=11)
ax.set_xlim(0, len(pattern))
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Layer Index', fontsize=12)
ax.set_title('Qwen 3.5: Hybrid Block Pattern (3:1 DeltaNet:Attention)', fontsize=14)
ax.set_yticks([])
plt.tight_layout()
plt.show()

from collections import Counter
counts = Counter(pattern)
for lt, c in counts.items():
    print(f"  {lt}: {c}/{len(pattern)} ({100*c/len(pattern):.0f}%)")

## 7. DeltaNet vs Mamba

Both are O(N), but fundamentally different:

| | Mamba (SSM) | DeltaNet |
|---|---|---|
| State | Vector $h \in \mathbb{R}^d$ | **Matrix** $W \in \mathbb{R}^{d \times d}$ |
| Update | $h = \bar{A}h + \bar{B}x$ | $W = \alpha W + \beta(v - Wk)k^T$ |
| Stores | Compressed sequence summary | **Key-value associations** |
| Correction | Difficult (state is opaque) | **Yes (delta rule corrects errors)** |
| Closer to | RNN | **Attention (learned KV cache)** |
| Throughput | Fast (simple state update) | Slower (matrix operations) |

DeltaNet's state matrix is a "learned, compressed KV cache" — conceptually closer
to attention than to an RNN. The 32% throughput penalty vs pure attention comes from
the sequential matrix updates that can't be parallelized across the sequence dimension.

## 8. MoE: 256 Experts, 9 Active

Qwen 3.5 uses extremely fine-grained MoE:
- **256 expert sub-networks** per FFN block
- **8 routed + 1 shared** active per token (3.5% activation)
- The shared expert captures universal features (syntax, grammar)
- Routed experts specialize: code syntax, math, languages, structured data

**Quantization warning:** With only 8 of 256 experts active, each expert's weight
quality matters enormously. Aggressive quantization (2-3 bit) degrades routing
accuracy. Q4 or higher recommended.

In [ ]:
# Parameter comparison: total vs active
d_model = 4096  # Qwen 3.5 35B-A3B scale
d_ff = 2048     # per expert
n_experts = 256
n_active = 9    # 8 routed + 1 shared

expert_params = 2 * d_model * d_ff  # one expert FFN
total_moe = n_experts * expert_params + expert_params  # 256 routed + 1 shared
active_moe = n_active * expert_params

print(f"Qwen 3.5 MoE per layer (35B-A3B scale):")
print(f"  Experts: {n_experts} routed + 1 shared")
print(f"  Active per token: {n_active} ({100*n_active/(n_experts+1):.1f}%)")
print(f"  Total FFN params: {total_moe/1e6:.0f}M")
print(f"  Active FFN params: {active_moe/1e6:.0f}M")
print(f"  Sparsity: {100*(1 - active_moe/total_moe):.1f}% of params unused per token")

## 9. Natively Multimodal

Unlike DeepSeek-V3 and Nemotron-3 (text-only), Qwen 3.5 is **multimodal by design**:

**DeepStack Vision Transformer:**
- Uses `Conv3d` for patch embeddings — treats video as a 3D volume (height × width × time)
- Merges features from **multiple ViT layers** (not just the last one) — captures both fine-grained and high-level visual details
- Early fusion: text, image, and video tokens are processed together from the start

This is conceptually different from "vision adapter" approaches (LLaVA, etc.) where
a pre-trained vision encoder is bolted onto a text LLM.

## 10. Full Model Assembly

In [ ]:
class Qwen35(nn.Module):
    """
    Scaled-down Qwen 3.5 architecture.
    Hybrid GatedDeltaNet + GQA Attention + MoE.
    """

    def __init__(self, vocab_size=1000, d_model=128, d_state=32,
                 n_q_heads=8, n_kv_heads=2, d_ff=64,
                 n_experts=32, top_k=4, n_cycles=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

        # Build 3:1 hybrid pattern
        self.layers = nn.ModuleList()
        self.layer_types = []

        for _ in range(n_cycles):
            # 3 DeltaNet + MoE blocks
            for _ in range(3):
                self.layers.append(GatedDeltaNet(d_model, d_state))
                self.layer_types.append('DeltaNet')
                self.layers.append(QwenMoELayer(d_model, d_ff, n_experts, top_k))
                self.layer_types.append('MoE')
            # 1 Attention + MoE block
            self.layers.append(GQABlock(d_model, n_q_heads, n_kv_heads))
            self.layer_types.append('Attention')
            self.layers.append(QwenMoELayer(d_model, d_ff, n_experts, top_k))
            self.layer_types.append('MoE')

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


# Create mini Qwen 3.5
model = Qwen35(
    vocab_size=1000, d_model=128, d_state=32,
    n_q_heads=8, n_kv_heads=2, d_ff=64,
    n_experts=32, top_k=4, n_cycles=2
)

tokens = torch.randint(0, 1000, (1, 16))
logits = model(tokens)

total_params = sum(p.numel() for p in model.parameters())
print(f"Mini Qwen 3.5:")
print(f"  Pattern: 3:1 (DeltaNet:Attention), 2 cycles")
print(f"  Layers: {len(model.layers)} ({Counter(model.layer_types)})")
print(f"  Input:  {tokens.shape}")
print(f"  Output: {logits.shape}")
print(f"  Total parameters: {total_params:,}")

## 11. Three Architectures Compared

| Innovation | DeepSeek-V3 | Nemotron-3 Super | **Qwen 3.5** |
|---|---|---|---|
| **Sequence layer** | MLA (attention) | Mamba-2 (SSM) | **GatedDeltaNet** |
| **State** | Full KV cache | State vector | **State matrix** |
| **Correction** | Perfect (re-attend) | Difficult | **Delta rule (corrective)** |
| **Complexity** | O(N²) | O(N) | **O(N)** |
| **Hybrid ratio** | 100% attention | 3:1 (Mamba:Attn) | **3:1 (DeltaNet:Attn)** |
| **MoE** | DeepSeekMoE (256, shared) | LatentMoE (latent routing) | **Fine-grained (256, shared)** |
| **Precision** | FP8 | FP4 | BF16 |
| **Context** | 128K | 1M | **1M** |
| **Multimodal** | No | No | **Yes (DeepStack ViT)** |

## 12. Which Paradigm Wins?

**It depends on the task:**

- **Precise recall** (code, formal logic): Attention (DeepSeek-V3). Nothing beats
  full KV cache for exact token lookup.
- **Very long context** (book summarization, code repos): Mamba (Nemotron-3).
  Linear complexity + constant-size state = practical 1M+ context.
- **Balanced tasks** (chat, reasoning, multimodal): DeltaNet (Qwen 3.5).
  Corrective state matrix gives better recall than Mamba at similar cost.

**The trend:** All three converge on **hybrid architectures** — using their primary
mechanism for most layers, with attention sprinkled in for precise recall.
The debate isn't "which is best" but "what ratio of each."

## 13. Key Takeaways

1. **The delta rule enables corrective state updates.** Unlike naive linear attention (accumulate only) or Mamba (opaque state), DeltaNet can fix wrong associations in its state matrix.

2. **Gating prevents state saturation.** Without forget/input gates, the state matrix grows unbounded. Sigmoid gates let the model selectively remember and forget.

3. **3:1 hybrid balances cost and quality.** DeltaNet handles 75% of layers at O(N) cost. GQA attention every 4th layer provides precise recall where needed.

4. **256-expert MoE gives extreme specialization.** With only 3.5% of experts active per token, each expert becomes highly specialized. The tradeoff: sensitivity to quantization.

5. **Three paradigms are converging on hybrid architectures.** Attention, SSMs, and DeltaNets each have strengths. The future is mixing them at different ratios per layer.

6. **DeltaNet is conceptually closest to attention.** Its state matrix is a "learned, compressed KV cache." This may explain why Qwen 3.5 shows strong performance on recall-heavy tasks compared to Mamba-based models.

### Prerequisite Notebooks

| Concept | Notebook |
|---|---|
| Attention & Transformers | `05_Papers/03_Attention_Is_All_You_Need` |
| MoE | `05_Papers/04_Mixture_of_Experts` |
| DeepSeek-V3 (MLA + MoE) | `07_Model_Replications/01_DeepSeek_V3` |
| Nemotron-3 (Mamba + MoE) | `07_Model_Replications/02_Nemotron3_Super` |

### Further Reading

- Qwen Team (2026). *Qwen 3.5 Technical Report.* https://github.com/QwenLM/Qwen3.5
- Yang et al. (2024). *Gated Delta Networks.* https://arxiv.org/abs/2412.06464
- Schlag et al. (2021). *Linear Transformers Are Secretly Fast Weight Programmers.* https://arxiv.org/abs/2102.11174